# Student dropout & academic success – full pipeline on Google Colab

Cost-sensitive soft-voting ensemble (LightGBM, CatBoost, XGBoost, ExtraTrees) on UCI dataset 697, compared with Islam et al. (2025) under three evaluation protocols.

**Before you start:** replace `YOUR-USERNAME` in the first code cell with your GitHub user name (or upload the repo folder to Colab / Google Drive instead). Runtime: *CPU is enough*; a full run takes hours, so start with the quick test (section 3).

## 1. Get the code and install packages

In [ ]:
REPO_URL = "https://github.com/YOUR-USERNAME/dropout-prediction.git"   # <-- change YOUR-USERNAME
import os
if not os.path.exists('/content/dropout-prediction'):
    !git clone $REPO_URL /content/dropout-prediction
%cd /content/dropout-prediction
!pip install -q -r requirements.txt
!python src/download_data.py   # only downloads if data/data.csv is missing

## 2. (Optional) save results to Google Drive so they survive a session reset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/dropout_results', exist_ok=True)
os.environ['RESULTS_DIR'] = '/content/drive/MyDrive/dropout_results/results'   # figures go to the parent folder
os.environ['N_JOBS'] = str(os.cpu_count())

## 3. Quick smoke test (reduced settings, a few minutes)

In [ ]:
!python src/run_all.py --fast

## 4. Full run (long). Run stages one at a time if the session may time out

In [ ]:
# !python src/run_all.py                    # everything
# or stage by stage:
# !python src/run_protocol_a.py              # 1 Islam et al. protocol (leaky), 10 seeds
# !python src/run_protocol_b.py              # 2
# !python src/run_protocol_c.py              # 3 leakage-free, Optuna, SHAP, fairness
# !python src/run_paper3_compare.py          # 4 proposed model v1/v2
# !python src/run_islam_fair.py              # 5 Islam pipeline without leakage
# !python src/run_ablation.py                # 6
# !python src/run_protocol_a_v2.py           # 7
# !python src/stats_islam.py                 # 8
# !python src/holdout_artifacts.py           # 9
# !python src/make_figures.py                # 10
# !python src/make_paper_figures.py          # 11

## 5. Look at the results

In [ ]:
import pandas as pd, os
R = os.environ.get('RESULTS_DIR', 'results')
print('Leakage-free tests vs Islam et al. (proposed model minus baseline):')
display(pd.read_csv(f'{R}/islam_fair_tests.csv').round(4))

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('paper_figures/*.png'))[:11]:
    print(f); display(Image(f, width=700))

## 6. Use the proposed model on your own split (example)
Train the cost-sensitive ensemble on 80 % of the students and score the other 20 %.

In [ ]:
import sys; sys.path.insert(0, "/content/dropout-prediction/src")
import numpy as np
from sklearn.model_selection import train_test_split
from common import *

df, X, y = load_data(); XF = add_features(X)                       # 36 raw -> 54 features
tr, te = train_test_split(np.arange(len(y)), test_size=0.2, stratify=y, random_state=42)
members = [Model("lgbm", BASE_LGB, "sqrt"),                         # "sqrt" = class weights (n / (K n_c))^0.5
           Model("cat", dict(iterations=500, depth=6, learning_rate=0.05, l2_leaf_reg=5), "sqrt"),
           Model("xgb", dict(n_estimators=300, max_depth=4, learning_rate=0.05, min_child_weight=3, reg_lambda=5), "sqrt"),
           Model("et", {}, "sqrt")]
members = [m.fit(XF.iloc[tr], y[tr]) for m in members]
proba = np.mean([m.predict_proba(XF.iloc[te]) for m in members], axis=0)   # soft vote
pred = proba.argmax(1)
print(metrics(y[te], pred))
print("first 3 students:", dict(zip(CLASS_NAMES, proba[0].round(3))))
